In [ ]:
import signal, time as _t
_T0 = _t.time()
def _cap(*a): raise TimeoutError("25 min cap hit after %.0fs" % (_t.time() - _T0))
signal.signal(signal.SIGALRM, _cap); signal.alarm(25 * 60)

import subprocess, os, sys, json, time, base64, io
def sh(c): print(c, flush=True); subprocess.run(c, shell=True, check=True)
sh("pip -q install 'transformers>=5.17' gguf>=0.10 accelerate git+https://github.com/huggingface/diffusers")
import torch, numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download, snapshot_download
import diffusers
from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel, GGUFQuantizationConfig
print("diffusers", diffusers.__version__, "torch", torch.__version__, flush=True)

REPO = "Qwen/Qwen-Image-2.1-Turbo"
BASE = snapshot_download(REPO, ignore_patterns=["transformer/*.safetensors", "vae/*.safetensors", "assets/*"])
gguf = hf_hub_download("unsloth/Qwen-Image-2.1-Turbo-GGUF", "qwen-image-2.1-turbo-Q4_K_S.gguf")
print("downloads done", round(time.time() - _T0), "s", flush=True)
D0, DT = torch.device("cuda:0"), torch.float16

exec(compile('"""Run the Qwen-Image(-Layered) transformer on GPUs without bf16 (Kaggle T4, P100, RTX 20xx) without NaNs.\n\nQwen-Image is trained for bf16; its hidden values exceed the fp16 limit (65504), so a plain fp16 run gives all-NaN\nlatents (ComfyUI also lists Qwen-Image as bf16/fp32 only). This keeps the fast fp16 tensor-core matmuls but holds\neverything else in fp32:\n\n  * every Linear: input scaled down to absmax 32, fp16 matmul, result scaled back in fp32\n    (falls back to a fp32 matmul for that call if the fp16 result is still non-finite)\n  * attention: q/k in fp16 (RMS-normalised, small), v scaled the same way (attention is linear in v)\n  * residual stream, norms, gates, embeddings: fp32\n  * the model\'s output is returned in fp16, so the diffusers pipeline is unchanged\n\nMeasured on Kaggle 2x T4 (Q4_K_S GGUF, 640 px, 4 layers): no NaN, 42 s/step, peak 8.8 / 8.4 GB.\n\n    from layerproto.fp16_mixed import apply_fp16_mixed\n    apply_fp16_mixed(transformer)        # once, after loading; idempotent\n"""\nimport types\n\nimport torch\nimport torch.nn.functional as F\n\nLINEAR_LIMIT = 32.0  # absmax the fp16 matmul input is scaled to (1024 was still too high: NaNs)\nATTN_LIMIT = 32.0\n\n\ndef _dequant(w):\n    if type(w).__name__ == "GGUFParameter":\n        from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n        return dequantize_gguf_tensor(w)\n    return w\n\n\ndef _mixed_linear(self, x):\n    x = x.float()\n    s = (x.abs().amax() / LINEAR_LIMIT).clamp_(min=1.0)\n    y = self._fp16_orig_forward((x / s).to(torch.float16)).float()   # = W x/s + b\n    b = getattr(self, "bias", None)\n    if not torch.isfinite(y).all():                                  # rare: this call in fp32\n        y = F.linear(x, _dequant(self.weight).float(), None if b is None else b.float())\n    elif b is None:\n        y = y * s\n    else:\n        b = b.float()\n        y = (y - b) * s + b\n    lora = getattr(self, "_fp16_lora", None)\n    if lora is not None:                                             # low-rank delta, same input scaling\n        A, B, k = lora\n        if A.device != x.device:                                     # module was moved (e.g. split over GPUs)\n            A, B = A.to(x.device), B.to(x.device)\n            self._fp16_lora = (A, B, k)\n        xs = (x / s).to(torch.float16)\n        y = y + F.linear(F.linear(xs, A), B).float() * (s * k)\n    return y\n\n\ndef _lora_pairs(sd):\n    """{module name: (A [r, in], B [out, r], alpha or None)} for common LoRA key layouts (PEFT / kohya / diffusers)."""\n    out = {}\n    for key in sd:\n        for a_suf, b_suf in ((".lora_A.weight", ".lora_B.weight"), (".lora_down.weight", ".lora_up.weight"),\n                             (".lora_A.default.weight", ".lora_B.default.weight")):\n            if key.endswith(a_suf):\n                base = key[: -len(a_suf)]\n                bkey = base + b_suf\n                if bkey in sd:\n                    alpha = sd.get(base + ".alpha")\n                    name = base\n                    for pre in ("base_model.model.", "diffusion_model.", "transformer.", "model."):\n                        if name.startswith(pre):\n                            name = name[len(pre):]\n                    out[name] = (sd[key], sd[bkey], None if alpha is None else float(alpha))\n    return out\n\n\ndef add_lora(model, path_or_state_dict, scale: float = 1.0):\n    """Attach a LoRA to an fp16_mixed-patched model (works with GGUF-quantised linears, nothing is merged).\n    Returns (matched modules, LoRA modules not found)."""\n    sd = path_or_state_dict\n    if isinstance(sd, str):\n        from safetensors.torch import load_file\n        sd = load_file(sd)\n    pairs = _lora_pairs(sd)\n    mods = dict(model.named_modules())\n    hit, miss = 0, []\n    for name, (A, B, alpha) in pairs.items():\n        m = mods.get(name)\n        if m is None or not hasattr(m, "_fp16_orig_forward"):\n            miss.append(name)\n            continue\n        dev = next((p.device for p in m.parameters()), torch.device("cpu"))\n        r = A.shape[0]\n        k = scale * ((alpha / r) if alpha else 1.0)\n        m._fp16_lora = (A.to(dev, torch.float16), B.to(dev, torch.float16), k)\n        hit += 1\n    return hit, miss\n\n\ndef remove_lora(model):\n    for _, m in model.named_modules():\n        if hasattr(m, "_fp16_lora"):\n            del m._fp16_lora\n\n\ndef _patch_attention():\n    import diffusers.models.transformers.transformer_qwenimage as tq\n    if getattr(tq, "_fp16_orig_attn", None) is not None:\n        return\n    orig = tq.dispatch_attention_fn\n\n    def attn_fp16(q, k, v, attn_mask=None, **kw):\n        s = (v.float().abs().amax() / ATTN_LIMIT).clamp_(min=1.0)\n        if attn_mask is not None and attn_mask.is_floating_point():\n            attn_mask = attn_mask.to(torch.float16)\n        o = orig(q.to(torch.float16), k.to(torch.float16), (v.float() / s).to(torch.float16), attn_mask=attn_mask, **kw)\n        return o.float() * s\n\n    tq._fp16_orig_attn = orig\n    tq.dispatch_attention_fn = attn_fp16\n\n\ndef apply_fp16_mixed(model):\n    """Patch a loaded QwenImageTransformer2DModel in place. Returns (linears patched, params moved to fp32)."""\n    _patch_attention()\n    n_lin = n_par = 0\n    for _, m in model.named_modules():\n        if "Linear" in type(m).__name__:\n            if not hasattr(m, "_fp16_orig_forward"):\n                m._fp16_orig_forward = m.forward\n            m.forward = types.MethodType(_mixed_linear, m)\n            n_lin += 1\n        else:\n            for _, p in m.named_parameters(recurse=False):\n                if p.dtype == torch.float16:\n                    p.data = p.data.float()\n                    n_par += 1\n    if not hasattr(model, "_fp16_orig_model_forward"):\n        model._fp16_orig_model_forward = model.forward\n\n        def fwd(*a, **k):\n            o = model._fp16_orig_model_forward(*a, **k)\n            if isinstance(o, tuple):\n                return (o[0].to(torch.float16),) + tuple(o[1:])\n            if hasattr(o, "sample"):\n                o.sample = o.sample.to(torch.float16)\n            return o\n\n        model.forward = fwd\n    return n_lin, n_par\n', "fp16_mixed", "exec"))
try:
    t = QwenImage21Transformer2DModel.from_single_file(gguf, config=BASE, subfolder="transformer",
            quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT)
    print("transformer: GGUF Q4_K_S", flush=True)
except Exception as e:  # diffusers GGUF loading for 2.1 is unconfirmed; NF4 is reported working (15 GB peak on a 5090 at 1024)
    import traceback; traceback.print_exc()
    sh("pip -q install bitsandbytes")
    from diffusers import BitsAndBytesConfig
    t = QwenImage21Transformer2DModel.from_pretrained(snapshot_download(REPO, allow_patterns=["transformer/*"]), subfolder="transformer",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DT), torch_dtype=DT)
    print("transformer: NF4 fallback", flush=True)
# diffusers' GGUF loader only dequantizes nn.Linear; other GGUF params keep packed bytes (run 1: 4096 vs 8192)
from diffusers.quantizers.gguf.utils import GGUFLinear, GGUFParameter, dequantize_gguf_tensor
_fixed = 0
for _m in t.modules():
    for _n, _p in list(_m.named_parameters(recurse=False)):
        if isinstance(_m, GGUFLinear) and _n == "weight":
            continue
        if isinstance(_p, GGUFParameter):
            setattr(_m, _n, torch.nn.Parameter(dequantize_gguf_tensor(_p).to(DT), requires_grad=False)); _fixed += 1
print("dequantized non-linear GGUF params:", _fixed, flush=True)
apply_fp16_mixed(t); t.to(D0).eval()

from transformers import AutoModelForImageTextToText
te = AutoModelForImageTextToText.from_pretrained(f"{BASE}/text_encoder", torch_dtype=DT, device_map="auto",
        max_memory={0: "5GiB", 1: "14GiB", "cpu": "24GiB"})
# stock 2.1 VAE gives NaNs in fp16, mostly in fully transparent regions; texture-fix VAE is fp16-safe drop-in
from diffusers import AutoencoderKLQwenImage21
vae = AutoencoderKLQwenImage21.from_pretrained("madebyollin/texture-fix-vae-for-qwen-image-2.1", torch_dtype=DT)
pipe = QwenImage21Pipeline.from_pretrained(BASE, transformer=t, text_encoder=te, vae=vae, torch_dtype=DT)
pipe.vae.to(D0)
_ep = pipe.encode_prompt
def _checked_encode(*a, **k):
    out = _ep(*a, **k)
    for o in (out if isinstance(out, tuple) else (out,)):
        if torch.is_tensor(o) and o.is_floating_point() and not torch.isfinite(o).all():
            print("[warn] non-finite prompt embeddings from the fp16 text encoder", flush=True)
    return out
pipe.encode_prompt = _checked_encode
type(pipe)._execution_device = property(lambda self: D0)
print("loaded", round(time.time() - _T0), "s; GiB", [round(torch.cuda.memory_allocated(i)/2**30, 1) for i in range(2)], flush=True)

os.makedirs("/kaggle/working/q21_out", exist_ok=True)
def alpha_stats(im):
    if im.mode != "RGBA": return f"mode {im.mode}"
    a = np.asarray(im.getchannel("A")); return f"RGBA, transparent {(a < 16).mean()*100:.0f}%, opaque {(a > 240).mean()*100:.0f}%"
def run(name, prompt, image=None, size=512, kv=True):
    try:
        torch.cuda.reset_peak_memory_stats(0); t0 = time.time()
        with torch.inference_mode():
            im = pipe(prompt=prompt, image=image, width=size, height=size, output_resolution=size, use_kv_cache=kv,
                      generator=torch.Generator(D0).manual_seed(42)).images[0]
        im.save(f"/kaggle/working/q21_out/{name}.png")
        print(f"{name}: {time.time()-t0:.0f}s, {alpha_stats(im)}, peak GiB {torch.cuda.max_memory_allocated(0)/2**30:.1f}", flush=True)
        return im
    except BaseException as e:
        import traceback; traceback.print_exc(); print("FAILED", name, flush=True)
        if isinstance(e, TimeoutError): raise

RGBA = "This is an RGBA image with transparency. {}. The image has alpha channel and the background is transparent."
fire = run("t2i_rgba_campfire", RGBA.format("A small campfire with crossed logs, impressionist oil painting style"))
run("t2i_rgba_campfire_nokv", RGBA.format("A small campfire with crossed logs, impressionist oil painting style"), kv=False)  # diffusers: KV cache not bit-exact in reduced precision
run("t2i_scene","An impressionist oil painting of a forest clearing at night with a campfire, pine trees and a full moon")
if fire is not None:
    run("edit_rgba_bluefire", RGBA.format("Make the flames blue, keep everything else the same"), image=[fire])
scene = Image.open(io.BytesIO(base64.b64decode("..."))).convert("RGB")
scene.save("/kaggle/working/q21_out/scene_input.png")
run("extract_campfire", RGBA.format("Extract only the campfire from the input image, keep its exact look"), image=[scene])
run("edit_scene_rain", "Make it a rainy night, keep the composition and style the same", image=[scene])
print("total", round(time.time() - _T0), "s", flush=True)